# Speed and Memory

Making training faster and fitting bigger models into memory. The rule: **measure first, then optimize the actual bottleneck.**

1. **Timing correctly** (GPU work is asynchronous)
2. **Profiling**: where does the time actually go?
3. **Data loading** as a bottleneck: `num_workers`, `pin_memory`
4. **Mixed precision** (`autocast`, `GradScaler`)
5. **Gradient accumulation**: big effective batch, small memory
6. **Gradient checkpointing**: trade compute for memory
7. **`torch.compile`**
8. **Cheap wins** checklist

> The code runs on CUDA, Apple MPS or CPU. Which speedups you *see* depends on hardware: mixed precision and `torch.compile` pay off mainly on modern CUDA GPUs. Numbers in the saved outputs come from the machine that ran this notebook; yours will differ. Where a technique is hardware-dependent, the notebook prints what it measured rather than assuming a win.

In [1]:
import time, copy
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

torch.manual_seed(0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print('device:', DEVICE, '| torch', torch.__version__)

def sync():
    if DEVICE == 'cuda': torch.cuda.synchronize()
    elif DEVICE == 'mps': torch.mps.synchronize()

def bench(fn, warmup=3, iters=10):
    """Median seconds per call. Warm up first; synchronize so we time the work, not the launch."""
    for _ in range(warmup): fn()
    sync(); times = []
    for _ in range(iters):
        t = time.perf_counter(); fn(); sync(); times.append(time.perf_counter() - t)
    return sorted(times)[len(times) // 2]

device: mps | torch 2.11.0


## 1. Timing correctly

GPU calls return immediately; the kernel runs later. Without `synchronize()`, a timer measures how long it took to *queue* the work. Warm-up matters too: the first calls include one-off setup (kernel compilation, memory allocation).

In [2]:
x = torch.randn(2048, 2048, device=DEVICE)
naive = time.perf_counter(); y = x @ x; naive = time.perf_counter() - naive      # no sync
print(f'naive timer   : {naive*1e3:8.3f} ms  (can be just the launch on GPU)')
print(f'bench (synced): {bench(lambda: x @ x)*1e3:8.3f} ms')

naive timer   :    4.737 ms  (can be just the launch on GPU)
bench (synced):    6.111 ms


## 2. Profile before optimizing

`torch.profiler` shows time per operator. A small model here; the habit is what matters.

In [3]:
from torch.profiler import profile, ProfilerActivity

model = nn.Sequential(nn.Linear(512, 2048), nn.ReLU(), nn.Linear(2048, 2048), nn.ReLU(), nn.Linear(2048, 10)).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), 1e-3)
xb, yb = torch.randn(256, 512, device=DEVICE), torch.randint(0, 10, (256,), device=DEVICE)

def step():
    opt.zero_grad(set_to_none=True)
    F.cross_entropy(model(xb), yb).backward(); opt.step()

for _ in range(3): step()
acts = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if DEVICE == 'cuda' else [])
with profile(activities=acts) as prof:
    for _ in range(5): step()
    sync()
sort_key = 'cuda_time_total' if DEVICE == 'cuda' else 'cpu_time_total'
print(prof.key_averages().table(sort_by=sort_key, row_limit=8, max_name_column_width=40))

----------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                    Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
----------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
               Optimizer.step#AdamW.step         7.32%     561.966us        45.23%       3.471ms     694.218us             5  
autograd::engine::evaluate_function: ...         0.43%      32.834us        13.51%       1.036ms      69.097us            15  
                         LinearBackward0         0.12%       9.501us        13.08%       1.004ms      66.908us            15  
                          aten::addcmul_        13.02%     999.005us        13.02%     999.005us      33.300us            30  
                   aten::linear_backward        12.68%     972.707us        12.95%     994.124us      66.275us 

/opt/anaconda3/lib/python3.12/site-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


Read it top-down: the biggest rows are where optimizing pays. (On MPS the profiler reports CPU-side launch time, not GPU kernel time, so the table is most informative on CUDA or CPU.) If the top row is a data-copy or `DataLoader` wait instead of matmul, the model is not your bottleneck (next section).

## 3. Data loading as a bottleneck

If the GPU waits for data, no model optimization helps. We simulate a dataset whose `__getitem__` is slow (like image decoding plus augmentation) and compare `num_workers`.

Settings:
- **`num_workers`**: load batches in parallel worker processes (0 = in the main process).
- **`pin_memory=True`**: page-locked host memory for faster host-to-GPU copies (CUDA).
- **`persistent_workers=True`**: keep workers alive between epochs.
- **`prefetch_factor`**: batches each worker prepares ahead.

In [4]:
# Workers are *spawned* on macOS/Windows and must import the Dataset class by name, which fails for
# classes defined inside a notebook. So write the class to a module file and import it.
import sys, tempfile, textwrap, pathlib
mod_dir = tempfile.mkdtemp(); sys.path.insert(0, mod_dir)
pathlib.Path(mod_dir, 'slow_ds.py').write_text(textwrap.dedent('''
    import time, torch
    from torch.utils.data import Dataset
    class SlowDataset(Dataset):
        def __len__(self): return 1024
        def __getitem__(self, i):
            time.sleep(0.005)                   # stand-in for decode + augment
            return torch.randn(32), i % 10
'''))
from slow_ds import SlowDataset

def epoch_time(workers):
    dl = DataLoader(SlowDataset(), batch_size=32, num_workers=workers,
                    persistent_workers=workers > 0)
    t = time.perf_counter()
    for xb, yb in dl: pass
    return time.perf_counter() - t

for w in [0, 2, 4]:
    print(f'num_workers={w}: {epoch_time(w):.2f}s per epoch')

num_workers=0: 6.31s per epoch


num_workers=2: 4.15s per epoch


num_workers=4: 3.10s per epoch


Rule of thumb: raise `num_workers` until the GPU is no longer waiting (start near the number of CPU cores, then measure). More workers than needed wastes RAM, and with tiny datasets worker start-up can cost more than it saves. On macOS/Windows workers are *spawned*, so the Dataset class must be importable from a module (not defined in a notebook cell, as the cell above works around) and scripts need an `if __name__ == '__main__':` guard. Hitting `Can't get attribute 'X' on <module '__main__'>` means exactly this.

## 4. Mixed precision

Run most math in a 16-bit format (half the memory traffic, and on tensor-core GPUs several times the matmul throughput), keep numerically sensitive ops (softmax, losses, reductions) in float32. `torch.autocast` picks the dtype per operation.

| dtype | range | precision | needs `GradScaler` |
|---|---|---|---|
| float32 | wide | high | no |
| float16 | **narrow** (underflows small gradients) | medium | **yes** |
| bfloat16 | wide (same as fp32) | lower | no |

`GradScaler` multiplies the loss by a large factor so small fp16 gradients do not underflow to zero, then unscales before the optimizer step. bfloat16 does not need it (CUDA Ampere and newer, many CPUs).

In [5]:
# What autocast does to dtypes (bfloat16 works on CPU, so this runs anywhere)
lin = nn.Linear(64, 64)
inp = torch.randn(8, 64)
with torch.autocast('cpu', dtype=torch.bfloat16):
    h = lin(inp)
    loss = F.cross_entropy(h, torch.randint(0, 64, (8,)))
print('params dtype          :', lin.weight.dtype, '  (master weights stay fp32)')
print('linear output dtype   :', h.dtype)
print('cross_entropy dtype   :', loss.dtype, '  (autocast keeps loss in fp32)')

params dtype          : torch.float32   (master weights stay fp32)
linear output dtype   : torch.bfloat16
cross_entropy dtype   : torch.float32   (autocast keeps loss in fp32)


In [6]:
# The standard AMP training step. fp16+GradScaler on CUDA; bf16 elsewhere (no scaler needed).
use_cuda = DEVICE == 'cuda'
amp_dtype = torch.float16 if use_cuda else torch.bfloat16
scaler = torch.amp.GradScaler('cuda', enabled=use_cuda)          # no-op when disabled

def amp_step(model, opt, xb, yb):
    opt.zero_grad(set_to_none=True)
    with torch.autocast(device_type=DEVICE if DEVICE != 'mps' else 'cpu', dtype=amp_dtype):
        loss = F.cross_entropy(model(xb), yb)
    scaler.scale(loss).backward()
    scaler.unscale_(opt)                                          # so clipping sees true grads
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt); scaler.update()
    return loss

m = copy.deepcopy(model); o = torch.optim.AdamW(m.parameters(), 1e-3)
if DEVICE == 'mps':
    print('MPS: autocast support is limited; timing AMP only on CUDA/CPU.')
else:
    t_fp32 = bench(lambda: (o.zero_grad(set_to_none=True), F.cross_entropy(m(xb), yb).backward(), o.step()))
    t_amp  = bench(lambda: amp_step(m, o, xb, yb))
    print(f'fp32 step: {t_fp32*1e3:.2f} ms | AMP ({amp_dtype}) step: {t_amp*1e3:.2f} ms | ratio {t_fp32/t_amp:.2f}x')

MPS: autocast support is limited; timing AMP only on CUDA/CPU.


Do not assume a speedup: small models and CPUs often show **no gain or a slowdown** (the casts cost more than the savings). The big wins come from large matmuls/convs on tensor-core GPUs, plus the memory saving that lets you raise the batch size.

## 5. Gradient accumulation

Want an effective batch of 256 but only 64 fits in memory? Run 4 micro-batches, **sum their gradients**, step once. Divide each micro-batch loss by the number of accumulation steps so the total matches the mean over the big batch.

Check it is numerically the same as one big batch:

In [7]:
def make(): 
    torch.manual_seed(1); return nn.Sequential(nn.Linear(20, 32), nn.ReLU(), nn.Linear(32, 3))
X, Y = torch.randn(64, 20), torch.randint(0, 3, (64,))

big = make()
F.cross_entropy(big(X), Y).backward()

acc = make(); ACC = 4
for xs, ys in zip(X.chunk(ACC), Y.chunk(ACC)):
    (F.cross_entropy(acc(xs), ys) / ACC).backward()               # grads accumulate in .grad

diff = max((a.grad - b.grad).abs().max().item() for a, b in zip(big.parameters(), acc.parameters()))
print('max gradient difference big-batch vs accumulated:', f'{diff:.2e}')

max gradient difference big-batch vs accumulated: 9.31e-09


In [8]:
# Training-loop shape: step the optimizer every ACC micro-batches
ACC = 4
opt = torch.optim.SGD(acc.parameters(), lr=0.1)
for i, (xs, ys) in enumerate(zip(X.chunk(16), Y.chunk(16))):
    (F.cross_entropy(acc(xs), ys) / ACC).backward()
    if (i + 1) % ACC == 0:
        opt.step(); opt.zero_grad(set_to_none=True)
print('stepped every', ACC, 'micro-batches')

stepped every 4 micro-batches


Caveats: BatchNorm statistics are computed per micro-batch (so accumulation is not exactly a big batch for models with BatchNorm); LayerNorm models are unaffected. Scale the learning rate with the *effective* batch size.

## 6. Gradient checkpointing

Backprop stores every intermediate activation, so activation memory grows with depth. **Checkpointing** stores only some and **recomputes** the rest during backward: roughly 30% more compute for far less activation memory. Use it when a deep model does not fit.

The gradients must be identical; only memory and time change.

In [9]:
from torch.utils.checkpoint import checkpoint

class Block(nn.Module):
    def __init__(self, d=256):
        super().__init__(); self.f = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, d))
    def forward(self, x): return x + self.f(x)

class Deep(nn.Module):
    def __init__(self, depth=12, use_ckpt=False):
        super().__init__(); self.blocks = nn.ModuleList(Block() for _ in range(depth)); self.ckpt = use_ckpt
    def forward(self, x):
        for b in self.blocks:
            x = checkpoint(b, x, use_reentrant=False) if self.ckpt and self.training else b(x)
        return x

torch.manual_seed(0); plain = Deep(use_ckpt=False)
ck = copy.deepcopy(plain); ck.ckpt = True
x = torch.randn(32, 256)
plain(x).pow(2).mean().backward(); ck(x).pow(2).mean().backward()
d = max((a.grad - b.grad).abs().max().item() for a, b in zip(plain.parameters(), ck.parameters()))
print('max grad difference, checkpointed vs plain:', f'{d:.2e}')

max grad difference, checkpointed vs plain: 0.00e+00


In [10]:
if DEVICE == 'cuda':
    for name, mdl in [('plain', plain), ('checkpointed', ck)]:
        m = copy.deepcopy(mdl).cuda(); xx = torch.randn(512, 256, device='cuda')
        torch.cuda.reset_peak_memory_stats(); m(xx).pow(2).mean().backward()
        print(f'{name:13s} peak memory: {torch.cuda.max_memory_allocated()/2**20:.1f} MiB')
else:
    print('Peak-memory comparison needs CUDA (torch.cuda.max_memory_allocated). Gradients matching is the part checked here.')

Peak-memory comparison needs CUDA (torch.cuda.max_memory_allocated). Gradients matching is the part checked here.


## 7. `torch.compile`

`torch.compile(model)` traces the model and fuses operations into optimized kernels (via TorchInductor). One line, but: the first call is slow (compiling), shapes changing between calls can trigger recompiles, and it needs a working compiler toolchain (a C++ compiler on CPU, Triton on CUDA). Guarded below so the notebook runs even when it is unavailable.

In [11]:
mdl = nn.Sequential(nn.Linear(512, 2048), nn.GELU(), nn.Linear(2048, 2048), nn.GELU(), nn.Linear(2048, 10)).to(DEVICE).eval()
inp = torch.randn(256, 512, device=DEVICE)
try:
    comp = torch.compile(mdl)
    t0 = time.perf_counter(); comp(inp); sync(); first = time.perf_counter() - t0
    with torch.no_grad():
        eager = bench(lambda: mdl(inp)); compiled = bench(lambda: comp(inp))
        same = torch.allclose(mdl(inp), comp(inp), atol=1e-4)
    print(f'first call (includes compile): {first:.2f}s')
    print(f'eager {eager*1e3:.2f} ms | compiled {compiled*1e3:.2f} ms | outputs match: {same}')
except Exception as e:
    print('torch.compile unavailable here:', type(e).__name__, str(e).splitlines()[0][:120])

first call (includes compile): 0.69s
eager 1.59 ms | compiled 1.69 ms | outputs match: True


Whether it wins depends on the model: big gains on fusion-friendly models (many small elementwise ops, e.g. Transformers) on GPU; little or none on a model that is already a few large matmuls. Always compare against eager, as above.

## 8. Cheap wins checklist

| Technique | Effect | Cost |
|---|---|---|
| `zero_grad(set_to_none=True)` | skips a memset; saves memory | none (default in recent PyTorch) |
| `torch.no_grad()` / `torch.inference_mode()` for eval | no graph, less memory, faster | none |
| `DataLoader(num_workers>0, pin_memory=True, persistent_workers=True)` | hides data loading | RAM |
| Bigger batch (to a point) | better hardware use | may need LR retune |
| Mixed precision (bf16/fp16) | speed + memory on tensor-core GPUs | numerical care |
| `.to(device, non_blocking=True)` with pinned memory | overlaps copy and compute | none |
| `channels_last` memory format for CNNs on GPU | faster convs with AMP | one line |
| `torch.backends.cudnn.benchmark = True` (fixed input sizes) | picks fastest conv algorithm | non-deterministic |
| Avoid `.item()` / `.cpu()` / printing inside the hot loop | each forces a GPU sync | none |
| Gradient accumulation | big effective batch | none |
| Gradient checkpointing | fits deeper models | ~30% more compute |
| `torch.compile` | kernel fusion | compile time, fragility |

**The `.item()` trap:** `loss.item()` every step blocks until the GPU finishes. Accumulate the loss as a tensor and read it once per epoch (or every N steps).

In [12]:
# The .item() trap, measured
m = nn.Linear(512, 512).to(DEVICE); inp = torch.randn(256, 512, device=DEVICE)
def with_item():
    for _ in range(50): m(inp).sum().item()
def without_item():
    tot = 0
    for _ in range(50): tot = tot + m(inp).sum()
    tot.item()
print(f'.item() every step: {bench(with_item)*1e3:.2f} ms | once at the end: {bench(without_item)*1e3:.2f} ms')

.item() every step: 18.54 ms | once at the end: 7.65 ms


## Exercises
1. Profile the CNN from `05_cv/01_cnn_mnist.ipynb`. Which operator dominates? Does it change if you raise the batch size?
2. Sweep `num_workers` and `batch_size` for the CIFAR-10 loader and plot images/second.
3. If you have a CUDA GPU: compare fp32, fp16+GradScaler and bf16 on a ResNet-18 step, in time and `max_memory_allocated`.
4. Use gradient accumulation to train with an effective batch of 512 on a model that only fits 64, and check the loss curve matches a true 512 batch.